# Base environment

This notebook is used to experiment with the simulation environment. Add new few features, check if it's working properly, examples, etc.

## Initialize

In [2]:
%load_ext autoreload
%autoreload 2
%matplotlib widget

In [3]:
from math import radians, pi, sin
import matplotlib.pyplot as plt
from tqdm.auto import trange
import xarray.ufuncs as xrf
import numba as nb
import numpy as np
from numba.experimental import jitclass
from IPython.display import display, JSON

from cw.context import time_it
from cw.filters import smooth_signal

from traj1.logger import Logger

from environment import LauncherV1SubOrbital, Stage, AP_NONE, AP_PITCH_CONTROL, AP_FLIGHT_PATH_CONTROL, AP_PITCH_RATE_CONTROL

Instructions for updating:
non-resource variables are not supported in the long term


## Create new Environment

Creates a new environment with an attached logger.

In [29]:
env = LauncherV1SubOrbital({
    "theta_e_random_window": None,
    "initial_theta_e": radians(15),
    "autopilot_mode": AP_FLIGHT_PATH_CONTROL
})
env

In [30]:
env.reset()

array([0.        , 0.26179939, 0.00083333])

In [31]:
logger = Logger()
logger.register_time_attribute(env.sim, "t")
logger.register(env.sim, "env", [
    "h", "i", "vie", "vie_hat", "reward",
    "gamma_e", "theta_e", "theta_i_dot",
    "ap_comm_gamma_e", "ap_comm_theta_e",
    "action_autopilot_mode", "action_autopilot_reference",
    "vii", "xii", "fii_thrust", "mass", "mass_dot",
    "end_at_apogee", "end_at_ground"
])

In [32]:
n_episodes = 1
max_time = 300

last_episode_result = None
batch_results = None

logger.episode_reset()
logger.batch_reset()
with time_it("sim"):
    for episiode_idx in range(n_episodes):
        observation = env.reset()
        for i in range(int(max_time / env.sim.integrator.h)):
            pitch_angle = 0.5 * pi # + 0.05 * sin(env.sim.t)
            observation, reward, done, info = env.step(pitch_angle)
            logger.step()
            if done:
                break
        last_episode_result = logger.episode_finish()
batch_results = logger.batch_finish()

sim: 0.08605377899948508 [s]


In [33]:
last_episode_result

<xarray.Dataset>
Dimensions:                         (d_2_0: 2, t: 464)
Coordinates:
  * t                               (t) float64 0.05 0.1 0.15 ... 23.15 23.2
Dimensions without coordinates: d_2_0
Data variables: (12/19)
    env_h                           (t) float64 1.0 0.9959 ... 828.5 832.7
    env_i                           (t) int64 1 2 3 4 5 ... 460 461 462 463 464
    env_vie                         (t, d_2_0) float64 7.704e-34 ... 82.77
    env_vie_hat                     (t, d_2_0) float64 9.482e-33 -1.0 ... 0.9999
    env_reward                      (t) float64 0.0 0.0 0.0 ... 0.0 0.0 8.43e-05
    env_gamma_e                     (t) float64 -1.571 -0.3149 ... 1.583 1.583
    ...                              ...
    env_xii                         (t, d_2_0) float64 1.064e-10 ... 1.738e+06
    env_fii_thrust                  (t, d_2_0) float64 6.568 1.76 ... 0.0 0.0
    env_mass                        (t) float64 1.2 1.2 1.199 ... 1.0 0.9998 1.0
    env_mass_dot                    (t) float64 -0.008665 -0.008665 ... 0.0 0.0
    env_end_at_apogee               (t) bool False False False ... False False
    env_end_at_ground               (t) bool True True True ... True True True

In [34]:
max(last_episode_result.env_reward)

<xarray.DataArray 'env_reward' ()>
array(8.43006426e-05)
Coordinates:
    t        float64 23.2

In [39]:
plt.figure()
last_episode_result.env_theta_i_dot.plot.line(x="t")
# abs(last_episode_result.env_gamma_e - 0.5 * pi).plot.line(x="t")
# last_episode_result.env_gamma_e.plot.line(x="t")
# last_episode_result.env_vie_hat.plot.line(x="t")
# last_episode_result.env_h.plot.line(x="t")

Canvas(toolbar=Toolbar(toolitems=[('Home', 'Reset original view', 'home', 'home'), ('Back', 'Back to previous …

In [36]:
plt.figure()
last_episode_result.env_h.plot.line(x="t")

Canvas(toolbar=Toolbar(toolitems=[('Home', 'Reset original view', 'home', 'home'), ('Back', 'Back to previous …

In [11]:
plt.figure()
plt.plot(last_episode_result.env_xii.values[:, 0], last_episode_result.env_xii.values[:, 1] - last_episode_result.env_xii.values[0, 1])

Canvas(toolbar=Toolbar(toolitems=[('Home', 'Reset original view', 'home', 'home'), ('Back', 'Back to previous …

In [39]:
env.reset()

array([0.        , 1.74225079])

In [498]:
a = pi * 0.5

env.sim.step((
    True,
    False,
    nb.int32(1),
    nb.float64(pi * 0.4),
))
env.sim_states_dict()

{'t': '4.959999999999939',
 'action_engine_on': 'True',
 'action_drop_stage': 'False',
 'action_autopilot_mode': '1',
 'action_autopilot_reference': '1.2566370614359172',
 'ap_comm_gamma_e': '1.2566370614359172',
 'ap_comm_theta_e': '1.6343987944770575',
 'gii': '[-2.51557786e-06 -1.62489013e+00]',
 'xii': '[2.68977215e+00 1.73740765e+06]',
 'vii': '[1.32699945 3.1147535 ]',
 'aii': '[0.05744953 0.67393445]',
 'tei': '[[-1.00000000e+00  1.54815259e-06]\n [ 1.54815259e-06  1.00000000e+00]]',
 'vie': '[-1.32699463  3.11475555]',
 'fii_thrust': '[0.08494602 3.39893868]',
 'theta_i': '1.545809604739566',
 'theta_i_dot': '0.08858764158489985',
 'theta_e': '1.5458111528921576',
 'mass': '1.47855504587152',
 'mass_dot': '-0.004332313965341487',
 'h': '7.646192363463342',
 'engine_on': 'True',
 'stage_state': '1',
 'stage_idx': '0',
 'stage_ignitions_left': '-1',
 'gamma_i': '1.1680478716984262',
 'gamma_e': '1.1680494198510174',
 'longitude': '1.570794778642305',
 'reward': '0.0',
 'score': '

{'t': '0.24000000000000007',
 'action_engine_on': 'True',
 'action_drop_stage': 'False',
 'action_autopilot_mode': '1',
 'action_autopilot_reference': '1.5707963267948966',
 'ap_comm_gamma_e': 'nan',
 'ap_comm_theta_e': 'nan',
 'gii': '[-9.94966983e-17 -1.62490440e+00]',
 'xii': '[1.06385069e-10 1.73740002e+06]',
 'vii': '[9.44265504e-18 1.54210260e-01]',
 'aii': '[3.93928797e-17 6.43334547e-01]',
 'tei': '[[-1.000000e+00  6.123234e-17]\n [ 6.123234e-17  1.000000e+00]]',
 'vie': '[-3.08148791e-33  1.54210260e-01]',
 'fii_thrust': '[2.08189956e-16 3.40000000e+00]',
 'theta_i': '1.5707963267948966',
 'theta_i_dot': '0.0',
 'theta_e': '1.5707963267948966',
 'mass': '1.4989602446483161',
 'mass_dot': '-0.004332313965341487',
 'h': '0.018369183409959078',
 'engine_on': 'True',
 'stage_state': '1',
 'stage_idx': '0',
 'stage_ignitions_left': '-3',
 'gamma_i': '1.5707963267948966',
 'gamma_e': '1.5707963267948966',
 'longitude': '1.5707963267948966',
 'reward': '0',
 'score': '0.0',
 'done': 

In [8]:
spec = [
    ("bar", nb.boolean),
    ("bas", nb.boolean),
    ("qux", nb.int32),
    ("qqq", nb.float64),
]
@jitclass(spec)
class Foo:
    def __init__(self):
        self.bar = 0
        self.bas = 0
        self.qux = 0
        self.qqq = 0
        
    def set(self, new_value):
        self.bar, self.bas, self.qux, self.qqq = new_value

In [9]:
foo = Foo()

foo.set((
    True,
    False,
    nb.int32(1),
    nb.float64(pi * 0.5)
))

In [10]:
foo.qqq

1.5707963267948966